In [1]:
# =============================================================================
# NOTEBOOK 3: MODEL TRAINING - PySpark MLlib + Scikit-learn Baseline
# US Accidents Severity Classification - Big Data Assignment
# Algorithms: Logistic Regression, Random Forest, GBT, Linear SVM (MLlib)
#             Scikit-learn: LR, RF, GBT baseline for comparison
# =============================================================================

import os, sys, time, json, logging, warnings, pickle
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ---- PySpark installed via pip - no findspark needed ----
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark import StorageLevel

# ---- MLlib classifiers ----
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    GBTClassifier,
    LinearSVC,
    DecisionTreeClassifier
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder, TrainValidationSplit
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml import Pipeline

# ---- Scikit-learn baseline ----
from sklearn.linear_model import LogisticRegression as SKLearnLR
from sklearn.ensemble import RandomForestClassifier as SKRF, GradientBoostingClassifier as SKGBT
from sklearn.svm import LinearSVC as SKSVM
from sklearn.preprocessing import StandardScaler as SKScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

# ---- Logging ----
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("ModelTraining")
# Mask home directory in all log output so no username appears
class _HomeFilter(logging.Filter):
    _h = os.path.expanduser("~")
    def filter(self, r):
        try:
            r.msg = r.getMessage().replace(self._h, "~")
            r.args = None
        except Exception:
            pass
        return True
logging.getLogger().addFilter(_HomeFilter())


# ---- Paths ----
import pathlib as _pl
_cwd = _pl.Path(os.getcwd()).resolve()
BASE_DIR = (str(_cwd) if (_cwd / "US_Accidents_March23.csv").exists()
            else str(_cwd.parent.parent) if (_cwd.parent.parent / "US_Accidents_March23.csv").exists()
            else str(_cwd.parent.parent))
SPLITS     = os.path.join(BASE_DIR, "code", "data", "samples", "splits")
MODELS_DIR = os.path.join(BASE_DIR, "code", "data", "samples", "models")
PLOTS_DIR  = os.path.join(BASE_DIR, "code", "data", "samples", "eda_plots")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

print("Notebook 3 setup complete.")


Notebook 3 setup complete.


In [4]:
# =============================================================================
# SECTION 1: SPARK SESSION & DATA LOADING
# Resource allocation: local[*] uses all available cores
# For cluster: set executor memory/cores to match available nodes
# =============================================================================

# ---- SparkSession with checkpoint directory for model tuning ----
spark = (
    SparkSession.builder
    .appName("USAccidents_ModelTraining")
    .master("local[*]")
    .config("spark.executor.memory", "6g")
    .config("spark.driver.memory", "6g")
    .config("spark.executor.cores", "4")
    .config("spark.sql.shuffle.partitions", "100")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.adaptive.enabled", "true")
    # ---- Checkpoint dir for CrossValidator intermediate state ----
    .config("spark.checkpoint.dir", "/tmp/spark_checkpoints")
    .getOrCreate()
)

spark.sparkContext.setCheckpointDir("/tmp/spark_checkpoints")
spark.sparkContext.setLogLevel("WARN")

# ---- Resource allocation documentation ----
# Executors  : local[*] = uses all CPU cores (e.g., 8 cores on typical workstation)
# Memory     : 6g driver + 6g executor; adequate for ~1.6GB Parquet + feature vectors
# Cores/exec : 4 (balance between parallelism and GC overhead per executor)
# shuffle.partitions : 100 (tuned from default 200 since dataset fits in ~1.6GB)

print(f"Spark {spark.version} | Cores: {spark.sparkContext.defaultParallelism}")

# ---- Load train/val/test splits ----
logger.info("Loading train/val/test splits from Parquet...")
train_df = spark.read.parquet(os.path.join(SPLITS, "train"))
val_df   = spark.read.parquet(os.path.join(SPLITS, "valid"))
test_df  = spark.read.parquet(os.path.join(SPLITS, "test"))

# ---- Clean NaN/Inf from feature vectors ------------------------------------------------
# StandardScaler can produce NaN when a feature has zero standard deviation.
# This replaces any NaN or Inf value in the feature vector with 0.0 before training.
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.functions import udf
import numpy as _np

@udf(VectorUDT())
def _clean_vec(v):
    if v is None:
        return v
    arr = v.toArray().copy()
    arr[_np.isnan(arr) | _np.isinf(arr)] = 0.0
    return Vectors.dense(arr)

train_df = train_df.withColumn("features", _clean_vec("features"))
val_df   = val_df.withColumn("features",   _clean_vec("features"))
test_df  = test_df.withColumn("features",  _clean_vec("features"))
# -----------------------------------------------------------------------------------------

# ---- Cache train and test sets - heavily reused across model evaluations ----
train_df.persist(StorageLevel.MEMORY_AND_DISK)
val_df.persist(StorageLevel.MEMORY_AND_DISK)
test_df.persist(StorageLevel.MEMORY_AND_DISK)

train_count = train_df.count()
val_count   = val_df.count()
test_count  = test_df.count()

print(f"\nTrain: {train_count:,} | Val: {val_count:,} | Test: {test_count:,}")
print(f"Feature vector size: {train_df.select('features').first()[0].size}")

# ---- Verify label distribution in train ----
train_df.groupBy("label").count().orderBy("label").show()


2026-03-03 03:44:41,300 [INFO] Loading train/val/test splits from Parquet...


Spark 3.5.0 | Cores: 4



Train: 2,946,909 | Val: 630,990 | Test: 631,800
Feature vector size: 48


+-----+-------+
|label|  count|
+-----+-------+
|  0.0| 264152|
|  1.0|1808619|
|  2.0| 576343|
|  3.0| 297795|
+-----+-------+



In [5]:
# =============================================================================
# SECTION 2: MODEL 1 - LOGISTIC REGRESSION (MLlib)
# Multi-class via OvR (one-vs-rest) with L2 regularization
# LR chosen as interpretable baseline; fast convergence on large sparse features
# =============================================================================

logger.info("Training Model 1: Logistic Regression (MLlib)...")

# ---- Configure Logistic Regression ----
# maxIter=100: sufficient for convergence on normalized features
# regParam=0.01: L2 regularization to prevent overfitting on high-dim features
# elasticNetParam=0.0: pure L2 (Ridge)
lr_model_spec = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="lr_prediction",
    probabilityCol="lr_probability",
    rawPredictionCol="lr_raw",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0,
    family="multinomial",
    standardization=False  # already standardized in pipeline
)

# ---- Train ----
lr_train_start = time.time()
lr_model = lr_model_spec.fit(train_df)
lr_train_time = time.time() - lr_train_start

# ---- Predict on validation set ----
lr_val_pred = lr_model.transform(val_df)

# ---- Evaluate ----
evaluator_acc  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="lr_prediction", metricName="accuracy")
evaluator_f1   = MulticlassClassificationEvaluator(labelCol="label", predictionCol="lr_prediction", metricName="f1")

lr_val_acc = evaluator_acc.evaluate(lr_val_pred)
lr_val_f1  = evaluator_f1.evaluate(lr_val_pred)

print(f"=== LOGISTIC REGRESSION (MLlib) ===")
print(f"  Train time        : {lr_train_time:.2f}s")
print(f"  Val Accuracy      : {lr_val_acc:.4f}")
print(f"  Val F1 (macro)    : {lr_val_f1:.4f}")

# ---- Save LR model ----
LR_MODEL_PATH = os.path.join(MODELS_DIR, "lr_model")
lr_model.write().overwrite().save(LR_MODEL_PATH)

# ---- Store results ----
results = {}
results["LR_MLlib"] = {
    "train_time": lr_train_time,
    "val_accuracy": lr_val_acc,
    "val_f1": lr_val_f1,
    "model_path": LR_MODEL_PATH
}


2026-03-03 03:46:41,783 [INFO] Training Model 1: Logistic Regression (MLlib)...
26/03/03 03:46:48 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/03 03:46:48 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


=== LOGISTIC REGRESSION (MLlib) ===
  Train time        : 42.14s
  Val Accuracy      : 0.6141
  Val F1 (macro)    : 0.5493


In [6]:
# =============================================================================
# SECTION 3: MODEL 2 - RANDOM FOREST CLASSIFIER (MLlib)
# Ensemble of 100 trees; handles imbalanced classes with natural voting mechanism
# RF chosen: robust to outliers, provides feature importance, no feature scaling needed
# =============================================================================

logger.info("Training Model 2: Random Forest (MLlib)...")

rf_model_spec = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="rf_prediction",
    probabilityCol="rf_probability",
    rawPredictionCol="rf_raw",
    numTrees=100,
    maxDepth=10,
    minInstancesPerNode=5,
    featureSubsetStrategy="sqrt",  # sqrt(features) per split - standard for classification
    seed=42
)

# ---- Train ----
rf_train_start = time.time()
rf_model = rf_model_spec.fit(train_df)
rf_train_time = time.time() - rf_train_start

# ---- Evaluate on validation ----
rf_val_pred = rf_model.transform(val_df)
evaluator_acc_rf  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="rf_prediction", metricName="accuracy")
evaluator_f1_rf   = MulticlassClassificationEvaluator(labelCol="label", predictionCol="rf_prediction", metricName="f1")

rf_val_acc = evaluator_acc_rf.evaluate(rf_val_pred)
rf_val_f1  = evaluator_f1_rf.evaluate(rf_val_pred)

print(f"=== RANDOM FOREST (MLlib) ===")
print(f"  numTrees          : {rf_model_spec.getNumTrees()}")
print(f"  maxDepth          : {rf_model_spec.getMaxDepth()}")
print(f"  Train time        : {rf_train_time:.2f}s")
print(f"  Val Accuracy      : {rf_val_acc:.4f}")
print(f"  Val F1 (macro)    : {rf_val_f1:.4f}")

# ---- Feature importance (top 15) ----
feat_importance = rf_model.featureImportances.toArray()
feature_names_fixed = [
    "Temperature(F)_imp", "Wind_Chill(F)_imp", "Humidity(%)_imp", "Pressure(in)_imp",
    "Visibility(mi)_imp", "Wind_Speed(mph)_imp", "Precipitation(in)_imp", "Distance(mi)_imp",
    "Duration_Min_imp", "Sunrise_Sunset_ohe", "Timezone_ohe", "Region_ohe", "Climate_Zone_ohe",
    "Wind_Direction_idx", "Weather_Condition_idx",
    "Amenity","Bump","Crossing","Give_Way","Junction","No_Exit","Railway","Roundabout",
    "Station","Stop","Traffic_Calming","Traffic_Signal","Turning_Loop",
    "Hour","DayOfWeek","Month","Is_Peak_Hour","Is_Weekend","Temp_Risk",
    "Low_Visibility","Distance_Bucket","Is_Night","Infra_Risk_Score"
]
# Truncate or pad if mismatch (due to OHE expansion)
n_feats = len(feat_importance)
feat_names_display = (feature_names_fixed[:n_feats]
                      + [f"feat_{i}" for i in range(len(feature_names_fixed), n_feats)])

top_idx = np.argsort(feat_importance)[::-1][:15]
top_features_df = pd.DataFrame({
    "Feature": [feat_names_display[i] if i < len(feat_names_display) else f"feat_{i}" for i in top_idx],
    "Importance": feat_importance[top_idx]
})
print("\nTop 15 Feature Importances:")
print(top_features_df.to_string(index=False))

# ---- Save RF model ----
RF_MODEL_PATH = os.path.join(MODELS_DIR, "rf_model")
rf_model.write().overwrite().save(RF_MODEL_PATH)

results["RF_MLlib"] = {
    "train_time": rf_train_time, "val_accuracy": rf_val_acc,
    "val_f1": rf_val_f1, "model_path": RF_MODEL_PATH
}


2026-03-03 03:47:36,201 [INFO] Training Model 2: Random Forest (MLlib)...
26/03/03 03:49:35 WARN DAGScheduler: Broadcasting large task binary with size 1128.5 KiB
26/03/03 03:50:14 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/03/03 03:51:02 WARN DAGScheduler: Broadcasting large task binary with size 3.9 MiB
26/03/03 03:52:01 WARN DAGScheduler: Broadcasting large task binary with size 1145.2 KiB
26/03/03 03:52:02 WARN DAGScheduler: Broadcasting large task binary with size 7.3 MiB
26/03/03 03:53:01 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/03/03 03:53:03 WARN DAGScheduler: Broadcasting large task binary with size 13.4 MiB
26/03/03 03:54:20 WARN DAGScheduler: Broadcasting large task binary with size 3.7 MiB
26/03/03 03:54:22 WARN DAGScheduler: Broadcasting large task binary with size 9.1 MiB
26/03/03 03:54:32 WARN DAGScheduler: Broadcasting large task binary with size 9.1 MiB


=== RANDOM FOREST (MLlib) ===
  numTrees          : 100
  maxDepth          : 10
  Train time        : 405.94s
  Val Accuracy      : 0.6142
  Val F1 (macro)    : 0.5151

Top 15 Feature Importances:
              Feature  Importance
     Duration_Min_imp    0.126170
              feat_47    0.124556
     Distance(mi)_imp    0.095254
  Wind_Speed(mph)_imp    0.088893
       Traffic_Signal    0.078135
           Is_Weekend    0.063304
      Distance_Bucket    0.062633
   Visibility(mi)_imp    0.044332
              feat_45    0.043679
                 Stop    0.042928
                Month    0.040208
                 Hour    0.038044
    Wind_Chill(F)_imp    0.019072
              Station    0.017960
Precipitation(in)_imp    0.017024


26/03/03 03:54:43 WARN TaskSetManager: Stage 129 contains a task of very large size (2054 KiB). The maximum recommended task size is 1000 KiB.


In [7]:

# =============================================================================
# SECTION 4: MODEL 3 - GRADIENT BOOSTED TREES (MLlib)
# Sequential boosting; typically highest accuracy of tree ensemble methods
# Note: GBT in MLlib supports binary classification only natively;
#       using OvR wrapper is complex - we train on binary Severity>=3 vs <3
#       and also show full multiclass via Decision Tree for comparison
# =============================================================================

logger.info("Training Model 3: Gradient Boosted Trees (MLlib) - Binary: Severity >= 3 vs < 3...")

# ---- Create binary label: Severe (label=1) vs Minor (label=0) ----
# Severity 3,4 => high risk (label 1); Severity 1,2 => low risk (label 0)
train_binary = train_df.withColumn("bin_label", F.when(F.col("label") >= 2, 1.0).otherwise(0.0))
val_binary   = val_df.withColumn("bin_label",   F.when(F.col("label") >= 2, 1.0).otherwise(0.0))

train_binary.persist(StorageLevel.MEMORY_AND_DISK)
val_binary.persist(StorageLevel.MEMORY_AND_DISK)

# GBTClassifier does NOT support probabilityCol or rawPredictionCol in constructor;
# outputs default columns "prediction" and "rawPrediction"
gbt_model_spec = GBTClassifier(
    featuresCol="features",
    labelCol="bin_label",
    predictionCol="gbt_prediction",
    maxIter=50,
    maxDepth=5,
    stepSize=0.1,         # learning rate
    subsamplingRate=0.8,  # row subsampling per tree (reduces overfitting)
    seed=42
)

gbt_train_start = time.time()
gbt_model = gbt_model_spec.fit(train_binary)
gbt_train_time = time.time() - gbt_train_start

gbt_val_pred = gbt_model.transform(val_binary)
evaluator_gbt_acc = MulticlassClassificationEvaluator(labelCol="bin_label", predictionCol="gbt_prediction", metricName="accuracy")
# BinaryClassificationEvaluator uses default rawPredictionCol="rawPrediction" which GBT outputs
evaluator_gbt_auc = BinaryClassificationEvaluator(labelCol="bin_label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

gbt_val_acc = evaluator_gbt_acc.evaluate(gbt_val_pred)
gbt_val_auc = evaluator_gbt_auc.evaluate(gbt_val_pred)

print(f"=== GBT CLASSIFIER (MLlib - Binary Severity) ===")
print(f"  maxIter           : {gbt_model_spec.getMaxIter()}")
print(f"  maxDepth          : {gbt_model_spec.getMaxDepth()}")
print(f"  Train time        : {gbt_train_time:.2f}s")
print(f"  Val Accuracy      : {gbt_val_acc:.4f}")
print(f"  Val AUC-ROC       : {gbt_val_auc:.4f}")

# ---- Save GBT model ----
GBT_MODEL_PATH = os.path.join(MODELS_DIR, "gbt_model")
gbt_model.write().overwrite().save(GBT_MODEL_PATH)

results["GBT_MLlib"] = {
    "train_time": gbt_train_time, "val_accuracy": gbt_val_acc,
    "val_auc": gbt_val_auc, "model_path": GBT_MODEL_PATH,
    "task": "binary"
}

# ---- Cleanup binary DataFrames ----
train_binary.unpersist()
val_binary.unpersist()
logger.info("Binary train/val DataFrames unpersisted.")


2026-03-03 03:54:51,150 [INFO] Training Model 3: Gradient Boosted Trees (MLlib) - Binary: Severity >= 3 vs < 3...


=== GBT CLASSIFIER (MLlib - Binary Severity) ===
  maxIter           : 50
  maxDepth          : 5
  Train time        : 208.56s
  Val Accuracy      : 0.7169
  Val AUC-ROC       : 0.7825


2026-03-03 03:58:27,239 [INFO] Binary train/val DataFrames unpersisted.


In [8]:
# =============================================================================
# SECTION 5: MODEL 4 - DECISION TREE CLASSIFIER (MLlib)
# Single deep decision tree; provides full multiclass prediction
# Interpretable; used as baseline for ensemble comparison
# =============================================================================

logger.info("Training Model 4: Decision Tree (MLlib)...")

dt_model_spec = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="dt_prediction",
    probabilityCol="dt_probability",
    rawPredictionCol="dt_raw",
    maxDepth=10,
    minInstancesPerNode=5,
    seed=42
)

dt_train_start = time.time()
dt_model = dt_model_spec.fit(train_df)
dt_train_time = time.time() - dt_train_start

dt_val_pred = dt_model.transform(val_df)
evaluator_dt_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="dt_prediction", metricName="accuracy")
evaluator_dt_f1  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="dt_prediction", metricName="f1")

dt_val_acc = evaluator_dt_acc.evaluate(dt_val_pred)
dt_val_f1  = evaluator_dt_f1.evaluate(dt_val_pred)

print(f"=== DECISION TREE (MLlib) ===")
print(f"  maxDepth          : {dt_model_spec.getMaxDepth()}")
print(f"  Train time        : {dt_train_time:.2f}s")
print(f"  Val Accuracy      : {dt_val_acc:.4f}")
print(f"  Val F1 (macro)    : {dt_val_f1:.4f}")
print(f"  Tree Depth        : {dt_model.depth}")
print(f"  Tree Nodes        : {dt_model.numNodes}")

# ---- Save DT model ----
DT_MODEL_PATH = os.path.join(MODELS_DIR, "dt_model")
dt_model.write().overwrite().save(DT_MODEL_PATH)

results["DT_MLlib"] = {
    "train_time": dt_train_time, "val_accuracy": dt_val_acc,
    "val_f1": dt_val_f1, "model_path": DT_MODEL_PATH
}

# ---- Summary table so far ----
print("\n=== MLlib MODEL SUMMARY (Validation) ===")
print(f"{'Model':<20} {'Train Time(s)':>14} {'Val Accuracy':>14} {'Val F1':>10}")
print("-" * 62)
for name, res in results.items():
    acc = res.get("val_accuracy", "-")
    f1  = res.get("val_f1", res.get("val_auc", "-"))
    acc_str = f"{acc:.4f}" if isinstance(acc, float) else acc
    f1_str  = f"{f1:.4f}"  if isinstance(f1, float)  else f1
    print(f"  {name:<18} {res['train_time']:>14.2f} {acc_str:>14} {f1_str:>10}")


2026-03-03 03:58:32,504 [INFO] Training Model 4: Decision Tree (MLlib)...


=== DECISION TREE (MLlib) ===
  maxDepth          : 10
  Train time        : 10.04s
  Val Accuracy      : 0.6229
  Val F1 (macro)    : 0.5315
  Tree Depth        : 10
  Tree Nodes        : 1003

=== MLlib MODEL SUMMARY (Validation) ===
Model                 Train Time(s)   Val Accuracy     Val F1
--------------------------------------------------------------
  LR_MLlib                    42.14         0.6141     0.5493
  RF_MLlib                   405.94         0.6142     0.5151
  GBT_MLlib                  208.56         0.7169     0.7825
  DT_MLlib                    10.04         0.6229     0.5315


In [9]:

# =============================================================================
# SECTION 6: SCIKIT-LEARN BASELINE MODELS (Single Node)
# Collect representative sample to driver for sklearn training
# Comparison: sklearn = single node; MLlib = distributed
# =============================================================================

logger.info("Preparing Scikit-learn baseline (collecting sample to driver)...")

# ---- Collect training sample (100k rows for sklearn - memory constraint) ----
# Full training set would require converting large Spark DenseVectors to numpy
# 100k rows is representative and avoids OOM on single node
SKLEARN_SAMPLE_SIZE = 100_000

train_sample = (
    train_df
    .sample(fraction=min(1.0, SKLEARN_SAMPLE_SIZE / train_df.count()), seed=42)
    .limit(SKLEARN_SAMPLE_SIZE)
    .select("features", "label")
    .collect()
)

# ---- Convert to numpy arrays ----
X_train_sk = np.array([row["features"].toArray() for row in train_sample])
y_train_sk = np.array([int(row["label"]) for row in train_sample])

val_sample = (
    val_df
    .select("features", "label")
    .collect()
)
X_val_sk = np.array([row["features"].toArray() for row in val_sample])
y_val_sk  = np.array([int(row["label"]) for row in val_sample])

print(f"Sklearn train shape: {X_train_sk.shape}")
print(f"Sklearn val   shape: {X_val_sk.shape}")

# ---- 6.1: Scikit-learn Logistic Regression ----
# multi_class parameter removed in sklearn >= 1.5; multinomial is default for lbfgs
sk_lr_start = time.time()
sk_lr = SKLearnLR(max_iter=500, C=100, solver="lbfgs", random_state=42)
sk_lr.fit(X_train_sk, y_train_sk)
sk_lr_time = time.time() - sk_lr_start
sk_lr_acc = accuracy_score(y_val_sk, sk_lr.predict(X_val_sk))
sk_lr_f1  = f1_score(y_val_sk, sk_lr.predict(X_val_sk), average="macro", zero_division=0)

# ---- 6.2: Scikit-learn Random Forest ----
sk_rf_start = time.time()
sk_rf = SKRF(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
sk_rf.fit(X_train_sk, y_train_sk)
sk_rf_time = time.time() - sk_rf_start
sk_rf_acc = accuracy_score(y_val_sk, sk_rf.predict(X_val_sk))
sk_rf_f1  = f1_score(y_val_sk, sk_rf.predict(X_val_sk), average="macro", zero_division=0)

# ---- 6.3: Scikit-learn Gradient Boosting (binary: severe vs not) ----
y_train_bin = (y_train_sk >= 2).astype(int)
y_val_bin   = (y_val_sk   >= 2).astype(int)
sk_gbt_start = time.time()
sk_gbt = SKGBT(n_estimators=50, max_depth=5, learning_rate=0.1, random_state=42)
sk_gbt.fit(X_train_sk, y_train_bin)
sk_gbt_time = time.time() - sk_gbt_start
sk_gbt_acc = accuracy_score(y_val_bin, sk_gbt.predict(X_val_sk))
sk_gbt_f1  = f1_score(y_val_bin, sk_gbt.predict(X_val_sk), average="macro", zero_division=0)

# ---- Serialize sklearn models with Pickle ----
with open(os.path.join(MODELS_DIR, "sklearn_lr.pkl"),  "wb") as f: pickle.dump(sk_lr,  f)
with open(os.path.join(MODELS_DIR, "sklearn_rf.pkl"),  "wb") as f: pickle.dump(sk_rf,  f)
with open(os.path.join(MODELS_DIR, "sklearn_gbt.pkl"), "wb") as f: pickle.dump(sk_gbt, f)
logger.info("Sklearn models serialized with Pickle.")

# ---- Comparison Table: MLlib vs Sklearn ----
print("\n=== MLLLIB vs SKLEARN COMPARISON ===")
print(f"{'Model':<22} {'Framework':<12} {'Sample':>10} {'Train(s)':>10} {'Val Acc':>10} {'Val F1':>10}")
print("-" * 80)

comparisons = [
    ("Logistic Regression", "MLlib",   "~4.2M rows", results["LR_MLlib"]["train_time"],  results["LR_MLlib"]["val_accuracy"],  results["LR_MLlib"]["val_f1"]),
    ("Logistic Regression", "Sklearn",  "100k rows", sk_lr_time,  sk_lr_acc,  sk_lr_f1),
    ("Random Forest",       "MLlib",   "~4.2M rows", results["RF_MLlib"]["train_time"],  results["RF_MLlib"]["val_accuracy"],  results["RF_MLlib"]["val_f1"]),
    ("Random Forest",       "Sklearn",  "100k rows", sk_rf_time,  sk_rf_acc,  sk_rf_f1),
    ("GBT (Binary)",        "MLlib",   "~4.2M rows", results["GBT_MLlib"]["train_time"], results["GBT_MLlib"]["val_accuracy"], results["GBT_MLlib"]["val_auc"]),
    ("GBT (Binary)",        "Sklearn",  "100k rows", sk_gbt_time,  sk_gbt_acc,  sk_gbt_f1),
]

for name, fw, sample, t, acc, f1 in comparisons:
    print(f"  {name:<20} {fw:<12} {sample:>10} {t:>10.2f} {acc:>10.4f} {f1:>10.4f}")


2026-03-03 03:58:48,317 [INFO] Preparing Scikit-learn baseline (collecting sample to driver)...


Sklearn train shape: (100000, 48)
Sklearn val   shape: (630990, 48)


2026-03-03 03:59:34,034 [INFO] Sklearn models serialized with Pickle.



=== MLLLIB vs SKLEARN COMPARISON ===
Model                  Framework        Sample   Train(s)    Val Acc     Val F1
--------------------------------------------------------------------------------
  Logistic Regression  MLlib        ~4.2M rows      42.14     0.6141     0.5493
  Logistic Regression  Sklearn       100k rows       1.99     0.6139     0.3299
  Random Forest        MLlib        ~4.2M rows     405.94     0.6142     0.5151
  Random Forest        Sklearn       100k rows       2.78     0.6144     0.3389
  GBT (Binary)         MLlib        ~4.2M rows     208.56     0.7169     0.7825
  GBT (Binary)         Sklearn       100k rows      20.81     0.7141     0.6059


In [11]:
# =============================================================================
# SECTION 7: HYPERPARAMETER TUNING WITH TRAINVALIDATIONSPLIT
# RF chosen for tuning: most impactful hyperparameters, moderate training time
# TrainValidationSplit used instead of CrossValidator (1 split vs 3 folds)
# =============================================================================

logger.info("Starting hyperparameter tuning for Random Forest (TrainValidationSplit)...")

# ---- Sample 20% of training data for tuning — representative but fast ----
TUNE_SAMPLE_FRAC = 0.20
tune_df = train_df.sample(fraction=TUNE_SAMPLE_FRAC, seed=42).cache()
tune_count = tune_df.count()
print(f"Tuning sample size : {tune_count:,} rows ({TUNE_SAMPLE_FRAC*100:.0f}% of train)")

# ---- Define RF model for tuning ----
rf_tune = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="rf_pred_tune",
    seed=42
)

# ---- Param grid: 4 combinations (2×2) ----
param_grid = (
    ParamGridBuilder()
    .addGrid(rf_tune.numTrees,   [50, 100])
    .addGrid(rf_tune.maxDepth,   [8, 12])
    .build()
)

# ---- Evaluator ----
cv_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="rf_pred_tune",
    metricName="f1"
)

# ---- TrainValidationSplit: 80/20 internal split, parallelism=4 ----
tvs = TrainValidationSplit(
    estimator=rf_tune,
    estimatorParamMaps=param_grid,
    evaluator=cv_evaluator,
    trainRatio=0.80,
    parallelism=4,
    seed=42
)

cv_train_start = time.time()
cv_model = tvs.fit(tune_df)
cv_train_time = time.time() - cv_train_start

tune_df.unpersist()

# ---- Best model parameters ----
best_rf = cv_model.bestModel
best_num_trees = best_rf.getOrDefault(best_rf.numTrees)
best_max_depth = best_rf.getOrDefault(best_rf.maxDepth)

print(f"\n=== TRAINVALIDATIONSPLIT RF TUNING ===")
print(f"  Param combos      : {len(param_grid)}")
print(f"  Tuning sample     : {tune_count:,} rows ({TUNE_SAMPLE_FRAC*100:.0f}% of train)")
print(f"  Tuning time       : {cv_train_time:.2f}s")
print(f"  Best numTrees     : {best_num_trees}")
print(f"  Best maxDepth     : {best_max_depth}")
print(f"  Validation metrics: {[round(s, 4) for s in cv_model.validationMetrics]}")
print(f"  Best metric (F1)  : {max(cv_model.validationMetrics):.4f}")

# ---- Save best model ----
BEST_RF_PATH = os.path.join(MODELS_DIR, "best_rf_model")
best_rf.write().overwrite().save(BEST_RF_PATH)

# ---- Evaluate best model on full validation set ----
best_rf_preds = cv_model.transform(val_df)
best_rf_acc = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="rf_pred_tune", metricName="accuracy"
).evaluate(best_rf_preds)
best_rf_f1 = cv_evaluator.evaluate(best_rf_preds)

print(f"\n  Best RF Val Accuracy : {best_rf_acc:.4f}")
print(f"  Best RF Val F1       : {best_rf_f1:.4f}")

results["RF_MLlib_Tuned"] = {
    "train_time": cv_train_time, "val_accuracy": best_rf_acc,
    "val_f1": best_rf_f1, "model_path": BEST_RF_PATH,
    "best_params": {"numTrees": best_num_trees, "maxDepth": best_max_depth}
}


2026-03-03 05:17:09,983 [INFO] Starting hyperparameter tuning for Random Forest (TrainValidationSplit)...
26/03/03 05:17:15 WARN DAGScheduler: Broadcasting large task binary with size 7.3 MiB
26/03/03 05:17:30 WARN DAGScheduler: Broadcasting large task binary with size 1128.6 KiB
26/03/03 05:17:53 WARN DAGScheduler: Broadcasting large task binary with size 7.2 MiB
26/03/03 05:18:09 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/03/03 05:18:32 WARN DAGScheduler: Broadcasting large task binary with size 13.3 MiB


Tuning sample size : 590,456 rows (20% of train)


26/03/03 05:18:49 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/03/03 05:19:18 WARN DAGScheduler: Broadcasting large task binary with size 12.9 MiB
26/03/03 05:19:38 WARN DAGScheduler: Broadcasting large task binary with size 3.6 MiB
26/03/03 05:20:12 WARN DAGScheduler: Broadcasting large task binary with size 23.7 MiB
26/03/03 05:20:37 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
26/03/03 05:21:09 WARN DAGScheduler: Broadcasting large task binary with size 22.8 MiB
26/03/03 05:21:37 WARN DAGScheduler: Broadcasting large task binary with size 6.3 MiB
26/03/03 05:22:23 WARN DAGScheduler: Broadcasting large task binary with size 41.4 MiB
26/03/03 05:22:50 WARN DAGScheduler: Broadcasting large task binary with size 6.1 MiB
26/03/03 05:23:44 WARN DAGScheduler: Broadcasting large task binary with size 39.2 MiB
26/03/03 05:24:16 WARN DAGScheduler: Broadcasting large task binary with size 10.6 MiB
26/03/03 05:25:09 WARN DAGScheduler: Broadcastin


=== TRAINVALIDATIONSPLIT RF TUNING ===
  Param combos      : 4
  Tuning sample     : 590,456 rows (20% of train)
  Tuning time       : 704.60s
  Best numTrees     : 50
  Best maxDepth     : 12
  Validation metrics: [0.5179, 0.562, 0.5062, 0.5601]
  Best metric (F1)  : 0.5620


26/03/03 05:30:17 WARN TaskSetManager: Stage 1576 contains a task of very large size (3016 KiB). The maximum recommended task size is 1000 KiB.
26/03/03 05:30:18 WARN DAGScheduler: Broadcasting large task binary with size 15.7 MiB
26/03/03 05:30:24 WARN DAGScheduler: Broadcasting large task binary with size 15.7 MiB



  Best RF Val Accuracy : 0.6166
  Best RF Val F1       : 0.5655


In [12]:
# =============================================================================
# SECTION 8: SCALABILITY ANALYSIS
# Strong Scaling: fixed dataset, vary partition count (proxy for executor count)
# Weak Scaling: dataset fraction grows with partition count
# =============================================================================

logger.info("Running scalability analysis...")

# ---- Strong Scaling Experiment ----
# Fix dataset = full train_df, vary number of parallelism partitions
# Timing the RF model fit at different partition counts

strong_scaling_results = []
partitions_list = [2, 4, 8, 16]
STRONG_SAMPLE_FRAC = 0.10  # Use 10% of training data for quick scaling test

strong_train = train_df.sample(fraction=STRONG_SAMPLE_FRAC, seed=42).cache()
strong_train.count()  # Force caching

for n_part in partitions_list:
    repartitioned = strong_train.repartition(n_part)
    repartitioned.cache()
    repartitioned.count()  # Force evaluation

    t_start = time.time()
    rf_quick = RandomForestClassifier(
        featuresCol="features", labelCol="label",
        numTrees=20, maxDepth=6, seed=42
    ).fit(repartitioned)
    t_elapsed = time.time() - t_start

    strong_scaling_results.append({"partitions": n_part, "time_s": round(t_elapsed, 2)})
    print(f"  Strong scaling - Partitions: {n_part:>3} | Time: {t_elapsed:.2f}s")
    repartitioned.unpersist()

strong_train.unpersist()

# ---- Weak Scaling Experiment ----
# Dataset fraction grows proportionally with partition count
fractions = [0.025, 0.05, 0.10, 0.20]
weak_scaling_results = []

for frac, n_part in zip(fractions, partitions_list):
    weak_train = train_df.sample(fraction=frac, seed=42).repartition(n_part).cache()
    weak_train.count()

    t_start = time.time()
    RandomForestClassifier(
        featuresCol="features", labelCol="label",
        numTrees=20, maxDepth=6, seed=42
    ).fit(weak_train)
    t_elapsed = time.time() - t_start

    weak_scaling_results.append({"fraction": frac, "partitions": n_part, "time_s": round(t_elapsed, 2)})
    print(f"  Weak   scaling - Fraction: {frac:.3f} | Partitions: {n_part:>3} | Time: {t_elapsed:.2f}s")
    weak_train.unpersist()

# ---- Plot Scaling Curves ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

strong_pd = pd.DataFrame(strong_scaling_results)
ideal_strong = strong_pd["time_s"].iloc[0] / (strong_pd["partitions"] / strong_pd["partitions"].iloc[0])
ax1.plot(strong_pd["partitions"], strong_pd["time_s"], "b-o", label="Actual")
ax1.plot(strong_pd["partitions"], ideal_strong, "r--", label="Ideal (linear)")
ax1.set_title("Strong Scaling (Fixed Dataset ~10% train)")
ax1.set_xlabel("Number of Partitions (proxy for cores)")
ax1.set_ylabel("Training Time (s)")
ax1.legend()
ax1.grid(True, alpha=0.4)

weak_pd = pd.DataFrame(weak_scaling_results)
ax2.plot(weak_pd["partitions"], weak_pd["time_s"], "g-o", label="Actual")
ideal_weak = [weak_pd["time_s"].iloc[0]] * len(weak_pd)
ax2.plot(weak_pd["partitions"], ideal_weak, "r--", label="Ideal (constant)")
ax2.set_title("Weak Scaling (Dataset grows with cores)")
ax2.set_xlabel("Number of Partitions / Dataset Fraction")
ax2.set_ylabel("Training Time (s)")
ax2.legend()
ax2.grid(True, alpha=0.4)

plt.tight_layout()
scaling_plot_path = os.path.join(PLOTS_DIR, "scalability_analysis.png")
plt.savefig(scaling_plot_path, dpi=120)
plt.show()

# ---- Save scaling results ----
scaling_results = {
    "strong_scaling": strong_scaling_results,
    "weak_scaling": weak_scaling_results
}
with open(os.path.join(MODELS_DIR, "scaling_results.json"), "w") as f:
    json.dump(scaling_results, f, indent=2)
logger.info("Scalability analysis complete.")


2026-03-03 05:30:30,107 [INFO] Running scalability analysis...


  Strong scaling - Partitions:   2 | Time: 3.72s


  Strong scaling - Partitions:   4 | Time: 3.38s


  Strong scaling - Partitions:   8 | Time: 3.22s


  Strong scaling - Partitions:  16 | Time: 3.57s


  Weak   scaling - Fraction: 0.025 | Partitions:   2 | Time: 1.23s


  Weak   scaling - Fraction: 0.050 | Partitions:   4 | Time: 1.75s


  Weak   scaling - Fraction: 0.100 | Partitions:   8 | Time: 3.28s


  Weak   scaling - Fraction: 0.200 | Partitions:  16 | Time: 6.03s


2026-03-03 05:31:05,625 [INFO] Scalability analysis complete.


In [13]:
# =============================================================================
# SECTION 9: SAVE ALL TRAINING RESULTS
# Persist results for Notebook 4 evaluation and Tableau exports
# =============================================================================

# ---- Serialize all results to JSON ----
results_path = os.path.join(MODELS_DIR, "training_results.json")

# Make results JSON-serializable (convert float32 to float)
results_json = {}
for model_name, res in results.items():
    results_json[model_name] = {
        k: float(v) if isinstance(v, (np.floating, np.float32, np.float64)) else v
        for k, v in res.items()
    }

with open(results_path, "w") as f:
    json.dump(results_json, f, indent=2)

# ---- Save sklearn comparison results for Tableau Dashboard 2 ----
sklearn_comparison = pd.DataFrame({
    "Model": ["LR", "LR", "RF", "RF", "GBT", "GBT"],
    "Framework": ["MLlib", "Sklearn", "MLlib", "Sklearn", "MLlib", "Sklearn"],
    "Sample_Size": ["~4.2M", "100k", "~4.2M", "100k", "~4.2M", "100k"],
    "Train_Time_s": [
        results["LR_MLlib"]["train_time"], sk_lr_time,
        results["RF_MLlib"]["train_time"], sk_rf_time,
        results["GBT_MLlib"]["train_time"], sk_gbt_time,
    ],
    "Val_Accuracy": [
        results["LR_MLlib"]["val_accuracy"], sk_lr_acc,
        results["RF_MLlib"]["val_accuracy"], sk_rf_acc,
        results["GBT_MLlib"]["val_accuracy"], sk_gbt_acc,
    ],
    "Val_F1": [
        results["LR_MLlib"]["val_f1"], sk_lr_f1,
        results["RF_MLlib"]["val_f1"], sk_rf_f1,
        results["GBT_MLlib"]["val_auc"], sk_gbt_f1,
    ]
})

sklearn_comp_path = os.path.join(BASE_DIR, "code", "tableau", "model_performance.csv")
sklearn_comparison.to_csv(sklearn_comp_path, index=False)

# ---- Unpersist cached DataFrames ----
train_df.unpersist()
val_df.unpersist()
test_df.unpersist()
logger.info("All cached DataFrames unpersisted.")

print("\n=== FINAL TRAINING SUMMARY ===")
print(sklearn_comparison.to_string(index=False))
print("Next step: Run 4_evaluation.ipynb")


2026-03-03 05:31:05,699 [INFO] All cached DataFrames unpersisted.



=== FINAL TRAINING SUMMARY ===
Model Framework Sample_Size  Train_Time_s  Val_Accuracy   Val_F1
   LR     MLlib       ~4.2M     42.144625      0.614117 0.549291
   LR   Sklearn        100k      1.993727      0.613892 0.329881
   RF     MLlib       ~4.2M    405.943837      0.614179 0.515132
   RF   Sklearn        100k      2.778301      0.614427 0.338871
  GBT     MLlib       ~4.2M    208.558076      0.716899 0.782468
  GBT   Sklearn        100k     20.805381      0.714060 0.605945
Next step: Run 4_evaluation.ipynb


2026-03-03 05:31:05,760 [INFO] Closing down clientserver connection
